# Create graph figure

### Import libraries

In [42]:
import igraph
import itertools
import pandas as pd
from collections import Counter, defaultdict
import networkx
from pyvis.network import Network

import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.lines import Line2D
import numpy as np
import json
import re

### Import data

In [26]:
df = pd.read_csv('../raw/databases_rna_sugarcane.csv')

In [27]:
# Create a mapping from time ontology IDs to human-readable names
# Used the 
# Mapping to a Units of measurement ontology (UO) for time/developmental stages
# This dictionary maps general time units (like months, days, hours) to their corresponding terms in the Units of measurement ontology (UO),
# https://www.ebi.ac.uk/ols4/ontologies/om
time2UO = {
    "UO_0000035": "months",
    "UO_0000033": "days",
    "UO_0000032": "hours",
}

# Mapping to Plant Ontology (PO) for developmental stages
# This dictionary maps specific developmental stage descriptions to their corresponding terms in the Plant Ontology (PO),
# https://bioportal.bioontology.org/ontologies/PO
time2PO = {
    "PO:0008037": "Seedling",
    "PO:0007089": "Elongation",
    "PO:0007134": "Vegetative",
    "PO:0007520": "Rooting",
    "PO:0007073": "Tillering",
    "PO:0007130": "Reproductive",
}
# Create a mapping from PECO IDs to human-readable names
# Used the Plant Experimental Conditions Ontology (PECO) database to find the corresponding names for each PECO ID
# https://bioportal.bioontology.org/ontologies/PECO
peco2name = {
    "PECO:0001062": "Control",
    " PECO:0001062": "Control",
    "PECO:0007404": "Drought",
    "PECO:0007241": "Plant Nutrient",
    "PECO:0007357": "Biotic Plant",
    "PECO:0007189": "Chemical",
    "PECO:0007191": "Abiotic Plant",
    "PECO:0007174": "Cold Temperature",
    "PECO:0007357,PECO:0007404": "Biotic + Drought",
    "PECO:0007333": "Insect Plant",
    "PECO:0007050": "Soil Texture",
    "PECO:0007078": "Light Quantity",
    "PECO:0007189,PECO:0007165": "Chemical + Plant Growth Hormone",
    "PECO:0007189,PECO:0007404": "Chemical + Drought",
    "PECO:0007185": "Salt",
    "PECO:0001037": "Oxidative Stress",
    "PECO:0007173": "High Temperature",
}
# Create a mapping from PO IDs to human-readable names
# Used the Plant Ontology (PO) database to find the corresponding names for each PO ID
# https://bioportal.bioontology.org/ontologies/PO
po2name = {
    "PO:0025034": "Leaf",
    "PO:0020142": "Stem Internode",
    "PO:0009047": "Stem",
    "PO:0009005": "Root",
    "PO:0000055": "Bud",
    "PO:0004709": "Auxiliary Bud",
    "PO:0009046": "Flower",
    "PO:0009013": "Meristem",
    "PO:0020121": "Lateral Root",
    "PO:0020040": "Leaf Base",
    "PO:0020137": "Leaf Apex",
    "PO:0020141": "Stem Node",
    "PO:0025223": "Vegetative Shoot Apex",
    "PO:0025178": "Stem Epidermis",
    "PO:0009011": "Plant Structure",
    "PO:0006109": "Pith",
}
# Using Plant Trait Ontology (TO) for trait mapping
# This dictionary maps specific trait descriptions related to pathogen resistance/susceptibility, stress tolerance/susceptibility,
# and other characteristics to their corresponding terms in the Plant Trait Ontology (TO),
# https://bioportal.bioontology.org/ontologies/PTO
trait2TO = {
    # Fungal pathogen resistance/susceptibility
    "TO:0000439": "Smut Susceptible",
    # Viral pathogen resistance/susceptibility
    "TO:0000148": "Viral Disease Response",
    # Bacterial pathogen resistance/susceptibility
    "TO:0000315": "Bacterial Disease Response",
    # Drought stress tolerance/susceptibility
    "TO:0000276": "Drought Tolerant",
    "TO:0000188": "Drought Sensitive",
    # Insect pest resistance/susceptibility
    "TO:0000261": "Highly Susceptible to S. frugiperda",
    # Nematod resistance/susceptibility
    "TO:0000384": "Susceptibility to Pratylenchus zeae",
    # Temperature stress tolerance/susceptibility
    "TO:0000303": "Low temperature stress response",
    # Nitrogen requirement
    "TO:0000011": "Nitrogen sensitive",
    # Sucrose related traits
    "TO:0000328": "Sucrose content",
    # Chemical sensitivity
    "TO:0000482": "Chemical stress response",
    # Other traits that don't fit neatly into the above categories
    "TO:0000326": "Leaf color",
}

In [28]:
ontology_map = {
    **time2UO,
    **time2PO,
    **peco2name,
    **po2name,
    **trait2TO,
}

def replace_ontology(value):
    if pd.isna(value):
        return value

    value = str(value).strip()

    # Exact match first
    if value in ontology_map:
        return ontology_map[value]

    # Handle comma-separated IDs
    if "," in value:
        return ", ".join(
            ontology_map.get(v.strip(), v.strip())
            for v in value.split(",")
        )

    return ontology_map.get(value, value)

# Only use to transform columns that contain ontology IDs, to avoid unintended replacements in other columns
ontology_cols = [col for col in df.columns if 'ontology' in col]
for col in ontology_cols:
    df[col] = df[col].apply(replace_ontology)

In [29]:
def create_edges_merge(df: pd.DataFrame) -> list:
    """
    Create edges based on shared ontology terms using a self-join approach.
    Args:
        df (pd.DataFrame): The input DataFrame containing BioProject and ontology columns.
    Returns:
        list: A list of tuples representing edges between BioProjects that share ontology terms.
    """
    edges = set()
    ontology_cols = [col for col in df.columns if 'ontology' in col]
    
    for col in ontology_cols:
        # Self-join on each ontology column
        merged = df[['BioProject', col]].merge(
            df[['BioProject', col]], 
            on=col, 
            how='inner'
        )
        # Filter to keep only run1 < run2 to avoid duplicates
        merged = merged[merged['BioProject_x'] < merged['BioProject_y']]
        edges.update(zip(merged['BioProject_x'], merged['BioProject_y']))
    
    return list(edges)

In [30]:
def create_edges_merge_by_ontology_type(df: pd.DataFrame) -> list:
    """
    Create edges based on shared ontology terms using a self-join approach.
    Args:
        df (pd.DataFrame): The input DataFrame containing BioProject and ontology columns.
    Returns:
        list: A list of tuples representing edges between BioProjects that share ontology terms.
    """
    edges = {}  # (BioProject_x, BioProject_y) -> list of ontology cols
    ontology_cols = [col for col in df.columns if 'ontology' in col]
    
    for col in ontology_cols:
        # Self-join on each ontology column
        merged = df[['BioProject', col]].merge(
            df[['BioProject', col]], 
            on=col, 
            how='inner'
        )
        # Filter to keep only run1 < run2 to avoid duplicates
        merged = merged[merged['BioProject_x'] < merged['BioProject_y']]
        
        for bp_x, bp_y in zip(merged['BioProject_x'], merged['BioProject_y']):
            key = (bp_x, bp_y)
            if key not in edges:
                edges[key] = set()
            edges[key].add(col)
    
    # Return list of (BioProject_x, BioProject_y, [ontologies])
    return [(a, b, sorted(cols)) for (a, b), cols in edges.items()]

In [31]:
def create_edges_merge_by_ontology_value(df: pd.DataFrame) -> list:
    """
    Create edges based on shared ontology terms using a self-join approach, grouping by ontology values.
    Args:
        df (pd.DataFrame): The input DataFrame containing BioProject and ontology columns.
    Returns:
        list: A list of tuples representing edges between BioProjects that share ontology terms, grouped by ontology values.
    """
    edges = {}
    ontology_cols = [col for col in df.columns if 'ontology' in col]
    
    for col in ontology_cols:
        merged = df[['BioProject', col]].dropna(subset=[col]).merge(  # drop NaN before merging
            df[['BioProject', col]].dropna(subset=[col]), 
            on=col, 
            how='inner'
        )
        merged = merged[merged['BioProject_x'] < merged['BioProject_y']]
        
        for bp_x, bp_y, ontology_value in zip(merged['BioProject_x'], merged['BioProject_y'], merged[col]):
            key = (bp_x, bp_y)
            if key not in edges:
                edges[key] = set()
            edges[key].add(ontology_value)
    
    return [(a, b, sorted(cols)) for (a, b), cols in edges.items()]

In [32]:
def create_edges_merge_ontology2node(df: pd.DataFrame) -> set:
    """
    Create edges between BioProjects and ontology terms, treating ontologies as nodes.
    Args:
        df (pd.DataFrame): The input DataFrame containing BioProject and ontology columns.
    Returns:
        list: A list of tuples representing edges between BioProjects and ontology terms.
    """
    edges = set()
    ontology_cols = [col for col in df.columns if 'ontology' in col]
    
    for col in ontology_cols:
        merged = df[['BioProject', col]].dropna(subset=[col]).drop_duplicates()
        
        for bp, ont in zip(merged['BioProject'], merged[col]):
            for o in ont.split(','):
                edges.add((bp, o.strip()))

    return edges

In [33]:
edges = create_edges_merge_ontology2node(df) # or other functions created above
print(f"Number of edges: {len(edges)}")

Number of edges: 371


In [34]:
graph = igraph.Graph.TupleList(
    edges,
    directed=False,
    vertex_name_attr='BioProject',
    edge_attrs=['ontologies'] # just for the functions that return ontologies as edge attributes
)

In [35]:
graph.summary()

'IGRAPH U--- 198 371 -- \n+ attr: BioProject (v), ontologies (e)'

In [36]:
def create_ontology_table(edges, *ontology_dicts):
    # Merge ontology dictionaries
    ontology_map = {}
    ontology_source = {}

    for source_name, d in ontology_dicts:
        ontology_map.update(d)
        ontology_source.update({k: source_name for k in d})

    # Compute degrees
    degree = Counter(ont for _, ont in edges)

    # Build table
    return pd.DataFrame({
        "Ontology Term": list(ontology_map.keys()),
        "Name": list(ontology_map.values()),
        "Ontology": [ontology_source[k] for k in ontology_map],
        "Node Degree": [degree.get(k, 0) for k in ontology_map]
    }).sort_values("Node Degree", ascending=False)

In [37]:
ontology_df = create_ontology_table(
    edges,
    ("UO", time2UO),
    ("PO (stage)", time2PO),
    ("PECO", peco2name),
    ("PO (anatomy)", po2name),
    ("TO", trait2TO)
)

In [20]:
# Export the ontology table to a CSV file
ontology_df.to_csv('../raw/ontology_table.csv', index=False)

In [ ]:
graph.es['ontologies'] = [o for o in graph.es['ontologies']] # just for the functions that return ontologies as edge attributes

# Export graphml to use in gephi or cytoscape
graph.write_graphml('../raw/graph_sugarcane.graphml')

### R1 answer

In [38]:
name2source = {}
for source_name, d in [
    ("UO", time2UO),
    ("PO (stage)", time2PO),
    ("PECO", peco2name),
    ("PO (anatomy)", po2name),
    ("TO", trait2TO),
]:
    for _id, name in d.items():
        name2source[name] = source_name

color_map = {
    "UO": "#4C72B0",
    "PO (stage)": "#DD8452",
    "PECO": "#55A868",
    "PO (anatomy)": "#8172B2",
    "TO": "#C44E52",
    "Other": "#999999",  # catches any term not found in the dicts above
}

In [39]:
# Build BioProject -> set(ontology terms) lookup from `edges`
bp2terms = defaultdict(set)
for bp, term in edges:
    bp2terms[bp].add(term)

term_degree = Counter(term for _, term in edges)  # #BioProjects per term

unmapped = [t for t in term_degree if t not in name2source]
if unmapped:
    print(f"NOTE: {len(unmapped)} terms not found in the ontology name "
          f"dicts (will be labeled 'Other'): {unmapped[:10]}"
          f"{' ...' if len(unmapped) > 10 else ''}")

In [41]:
# Project to a term-term network: edge weight = # BioProjects that share BOTH terms

term_edge_weight = Counter()
for bp, terms in bp2terms.items():
    for t1, t2 in itertools.combinations(sorted(terms), 2):
        term_edge_weight[(t1, t2)] += 1

G = nx.Graph()
for term, deg in term_degree.items():
    G.add_node(
        term,
        label=term,  # already a human-readable name
        source=name2source.get(term, "Other"),
        n_projects=deg,
    )

for (t1, t2), w in term_edge_weight.items():
    if t1 in G.nodes and t2 in G.nodes:
        G.add_edge(t1, t2, weight=w)

print(f"Collapsed graph: {G.number_of_nodes()} ontology-term nodes, "
      f"{G.number_of_edges()} edges")

# Optional: drop very low-degree terms to declutter further, keep only terms used by >= 3 BioProjects
MIN_PROJECTS = 3
keep = [n for n, d in G.nodes(data=True) if d["n_projects"] >= MIN_PROJECTS]
G = G.subgraph(keep).copy()
print(f"After filtering (>= {MIN_PROJECTS} BioProjects): "
      f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Export for Gephi
nx.write_graphml(G, "../raw/figure2_ontology_terms.graphml")
print(" - ../raw/figure2_ontology_terms.graphml")

Collapsed graph: 53 ontology-term nodes, 190 edges
After filtering (>= 3 BioProjects): 22 nodes, 102 edges
 - ../raw/figure2_ontology_terms.graphml


In [43]:
#  Static figure (matplotlib)

colorblind_friendly_colors = {
    "UO": "#56B4E9",           # Sky Blue
    "PO (stage)": "#E69F00",   # Orange
    "PECO": "#009E73",         # Bluish Green
    "PO (anatomy)": "#CC79A7", # Reddish Purple
    "TO": "#F0E442",           # Yellow
    "Other": "#999999",        # Grey
}
color_map = colorblind_friendly_colors

fig, ax = plt.subplots(figsize=(16, 14), dpi=300)
ax.set_facecolor("#FFFFFF")

# Kamada-Kawai layout
pos = nx.kamada_kawai_layout(G, weight="weight")

min_deg = min(G.nodes[n]["n_projects"] for n in G.nodes) if G.nodes else 1
max_deg = max(G.nodes[n]["n_projects"] for n in G.nodes) if G.nodes else 1

node_sizes = [
    180 + 950 * ((G.nodes[n]["n_projects"] - min_deg) / (max_deg - min_deg + 1e-5))
    for n in G.nodes
]
node_colors = [color_map.get(G.nodes[n]["source"], "#999999") for n in G.nodes]

# Edge widths proportional to weight
max_w = max((w["weight"] for _, _, w in G.edges(data=True)), default=1)
edge_widths = [0.5 + 2.2 * (G[u][v]["weight"] / max(1, max_w)) for u, v in G.edges]

# Curved edges (arc3 rad=0.15)
nx.draw_networkx_edges(
    G, pos, ax=ax,
    width=edge_widths,
    edge_color="#8c8c8c",
    alpha=0.40,
    connectionstyle="arc3,rad=0.15",
    arrows=True,
    arrowstyle="-",
    node_size=node_sizes,
)

# Nodes with crisp dark borders and high opacity
nx.draw_networkx_nodes(
    G, pos, ax=ax,
    node_size=node_sizes,
    node_color=node_colors,
    edgecolors="#1a1a1a",
    linewidths=0.9,
    alpha=0.95,
)

# Typography with white halos
labels = {n: G.nodes[n]["label"] for n in G.nodes}
text_objs = nx.draw_networkx_labels(
    G, pos, labels=labels, ax=ax,
    font_size=10, font_family="sans-serif",
    font_color="#1a1a1a",
)
for _, t in text_objs.items():
    t.set_path_effects([pe.withStroke(linewidth=3.2, foreground="#ffffffea")])

# Single-line bottom legend
present_sources = sorted({G.nodes[n]["source"] for n in G.nodes})
legend_handles = [
    Line2D([0], [0], marker="o", color="w", label=src,
           markerfacecolor=color_map[src], markersize=11,
           markeredgecolor="#1a1a1a", markeredgewidth=0.9)
    for src in present_sources
]
ax.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.04),
    ncol=len(legend_handles),
    frameon=True, fancybox=True,
    framealpha=0.95, edgecolor="#cccccc",
    fontsize=11,
    title="Ontology Category", title_fontsize=12,
    handletextpad=0.5, columnspacing=1.5,
)

ax.set_title(
    "Major Ontology Terms across Sugarcane RNA-seq BioProjects",
    fontsize=16, fontweight="bold", color="#111111", pad=18,
)

ax.text(
    0.5, 1.005,
    f"{G.number_of_nodes()} terms shared by ≥ 3 BioProjects · {G.number_of_edges()} co-occurrence edges",
    transform=ax.transAxes,
    ha="center", va="bottom",
    fontsize=10, style="italic", color="#555555",
)

ax.axis("off")
fig.tight_layout()

plt.savefig("../raw/figure2_ontology_terms.png",
            dpi=300, bbox_inches="tight", facecolor="#FFFFFF")
plt.savefig("../raw/figure2_ontology_terms.pdf",
            bbox_inches="tight", facecolor="#FFFFFF")
plt.close(fig)

In [44]:
#  Full BioProject <-> ontology-term network — searchable HTML

net = Network(
    height="100vh", width="100%", bgcolor="#ffffff",
    font_color="#222222", notebook=False,
    filter_menu=False, select_menu=False,
    cdn_resources="in_line",
)

# Explicit vis.js options
vis_options = {
    "nodes": {
        "borderWidthSelected": 3,
        "shape": "dot"
    },
    "edges": {
        "smooth": {
            "enabled": True,
            "type": "continuous",
            "roundness": 0.15
        },
        "color": {
            "color":     "#D5D5D5",
            "highlight": "#56B4E9",
            "hover":     "#888888",
            "opacity":   0.6
        },
        "width": 1,
        "selectionWidth": 2.2
    },
    "physics": {
        "enabled": True,
        "barnesHut": {
            "gravitationalConstant": -3500,
            "centralGravity": 0.18,
            "springLength": 120,
            "springConstant": 0.035,
            "damping": 0.45,
            "avoidOverlap": 0.6
        }
    }
}

net.set_options(json.dumps(vis_options))

all_bioprojects = df["BioProject"].dropna().unique()

# BioProject nodes (all included)
for bp in all_bioprojects:
    terms = bp2terms.get(bp, set())
    term_count = len(terms)
    terms_str = ", ".join(sorted(terms)) if terms else "None"
    net.add_node(
        bp,
        label=bp,
        color={
            "border": "#888888",
            "background": "#F0F0F0",
            "highlight": {"border": "#333333", "background": "#D8D8D8"},
        },
        size=8,
        group="BioProject",
        shape="dot",
        borderWidth=1,
        title=(
            f"<div style='font-family:Arial;padding:4px;'>"
            f"<b style='color:#1a1a1a;font-size:14px;'>{bp}</b><br>"
            f"<span style='color:#666;font-size:12px;'>BioProject</span><br><br>"
            f"<b>Associated Terms:</b> {term_count}<br>"
            f"<b>Terms:</b><br>"
            f"<span style='font-size:11px;color:#444;'>{terms_str}</span>"
            f"</div>"
        ),
    )

# Ontology term nodes (sized by log(degree), colored by source)
for term in term_degree:
    source = name2source.get(term, "Other")
    cat_color = color_map.get(source, "#999999")
    net.add_node(
        term,
        label=term,
        color={
            "border": "#1a1a1a",
            "background": cat_color,
            "highlight": {"border": "#000000", "background": cat_color},
        },
        size=15 + np.log1p(term_degree[term]) * 8,
        group=source,
        shape="dot",
        borderWidth=1.5,
        borderWidthSelected=3,
        font={"size": 16, "face": "sans-serif", "color": "#111111"},
        title=(
            f"<div style='font-family:Arial;padding:4px;'>"
            f"<b style='color:#1a1a1a;font-size:14px;'>{term}</b><br>"
            f"<span style='color:#666;font-size:12px;'>"
            f"Ontology Term ({source})</span><br><br>"
            f"<b>Connected BioProjects:</b> {term_degree[term]}"
            f"</div>"
        ),
    )

# Edges (subtle weak lines to prevent visual clutter) 
for bp, term in edges:
    net.add_edge(
        bp, term,
        color={"color": "#E5E5E5", "highlight": "#56B4E9", "hover": "#888888", "opacity": 0.45},
        width=0.8,
        hoverWidth=1.8, selectionWidth=2.2,
    )

html_path = "../raw/figure2_supplementary_interactive.html"
net.write_html(html_path, notebook=False)

# Directly inject node colors into the Vis.js DataSet JSON string.
with open(html_path, "r", encoding="utf-8") as f:
    html_content = f.read()

def inject_node_colors(match):
    nodes_data = json.loads(match.group(1))
    for n in nodes_data:
        g = n.get("group", "Other")
        col = color_map.get(g, "#999999")
        n["color"] = {
            "background": col,
            "border": "#1a1a1a" if g != "BioProject" else "#888888",
            "highlight": {"background": col, "border": "#000000"}
        }
    return "nodes = new vis.DataSet(" + json.dumps(nodes_data) + ");"

html_content = re.sub(r'nodes = new vis\.DataSet\((\[.*?\])\);', inject_node_colors, html_content)

# Convert Python color_map into JSON string for JS legend synchronization
js_palette_json = json.dumps({**color_map, "BioProject": "#F0F0F0"})

custom_ui = f"""
<style>
  html, body {{
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    margin: 0; padding: 0; height: 100%;
  }}
  #header {{
    position: absolute; top: 0; left: 0; right: 0; height: 58px;
    background: linear-gradient(135deg, #1a1a1a, #2c2c2c);
    color: #fff; padding: 0 24px;
    display: flex; align-items: center; justify-content: space-between;
    box-shadow: 0 2px 8px rgba(0,0,0,0.15); z-index: 1000;
  }}
  #header h1 {{ margin: 0; font-size: 17px; font-weight: 500; letter-spacing: 0.2px; }}
  #header .stats {{ font-size: 12px; color: #cfcfcf; }}

  #controls {{
    position: absolute; top: 78px; left: 14px; width: 260px;
    background: rgba(255,255,255,0.98);
    border: 1px solid #e0e0e0; border-radius: 8px;
    padding: 14px 16px; box-shadow: 0 2px 8px rgba(0,0,0,0.08);
    z-index: 999; max-height: calc(100vh - 100px); overflow-y: auto;
  }}
  #controls h3 {{
    margin: 0 0 10px 0; font-size: 12px; color: #1a1a1a;
    text-transform: uppercase; letter-spacing: 0.6px;
    border-bottom: 1px solid #eee; padding-bottom: 6px;
  }}
  #controls h3:not(:first-child) {{ margin-top: 16px; }}
  .search-wrapper {{ position: relative; width: 100%; }}
  #searchBox {{
    width: 100%; padding: 8px 12px; border: 1px solid #ccc;
    border-radius: 4px; font-size: 13px; box-sizing: border-box;
    transition: border-color 0.2s;
  }}
  #searchBox:focus {{ outline: none; border-color: #56B4E9; box-shadow: 0 0 0 2px rgba(86,180,233,0.2); }}
  #searchResults {{
    position: absolute; top: 38px; left: 0; right: 0;
    background: #fff; border: 1px solid #ccc; border-top: none;
    border-radius: 0 0 4px 4px; max-height: 180px; overflow-y: auto;
    box-shadow: 0 4px 8px rgba(0,0,0,0.12); display: none; z-index: 1001;
  }}
  .search-item {{
    padding: 7px 12px; font-size: 12.5px; cursor: pointer; color: #333;
    display: flex; justify-content: space-between; align-items: center;
    border-bottom: 1px solid #f4f4f4;
  }}
  .search-item:hover {{ background: #f0f7fc; }}
  .search-item .cat-tag {{ font-size: 10px; padding: 2px 6px; border-radius: 3px; color: #fff; font-weight: bold; }}

  select.attribute-select {{
    width: 100%; padding: 8px 10px; border: 1px solid #ccc;
    border-radius: 4px; font-size: 13px; background: #fff; color: #333;
    cursor: pointer; box-sizing: border-box;
  }}
  select.attribute-select:focus {{ outline: none; border-color: #56B4E9; }}

  .legend-item {{
    display: flex; align-items: center; padding: 6px 8px;
    cursor: pointer; border-radius: 4px; transition: background 0.2s;
    user-select: none;
  }}
  .legend-item:hover {{ background: #f4f4f4; }}
  .legend-item.active {{ background: #eaf3fa; }}
  .legend-item.inactive {{ opacity: 0.45; }}
  .legend-dot {{
    width: 14px; height: 14px; border-radius: 50%;
    margin-right: 10px; border: 1px solid #333; flex-shrink: 0;
  }}
  .legend-label {{ font-size: 13px; color: #333; }}
  .legend-count {{ margin-left: auto; font-size: 11px; color: #888; font-weight: bold; }}
  .btn {{
    width: 100%; padding: 7px 10px; margin-top: 6px;
    border: 1px solid #ccc; border-radius: 4px;
    background: #f8f9fa; color: #333; cursor: pointer;
    font-size: 12px; transition: background 0.2s, border-color 0.2s;
  }}
  .btn:hover {{ background: #e8e8e8; border-color: #bbb; }}

  #node-info-panel {{
    position: absolute; top: 78px; right: 14px; width: 320px;
    background: rgba(255,255,255,0.98);
    border: 1px solid #e0e0e0; border-radius: 8px;
    padding: 18px 20px; box-shadow: 0 2px 8px rgba(0,0,0,0.08);
    display: none; z-index: 999;
  }}
  #node-info-panel h3 {{
    margin: 0 0 10px 0; color: #1a1a1a; font-size: 15px;
    border-bottom: 2px solid #56B4E9; padding-bottom: 8px;
    word-break: break-word;
  }}
  #node-info-panel .info-body {{ color: #444; line-height: 1.55; font-size: 13px; }}
  #node-info-panel .close-btn {{
    margin-top: 14px; padding: 7px 10px;
    background: #f8f9fa; border: 1px solid #ccc; border-radius: 4px;
    cursor: pointer; color: #333; font-weight: 500; width: 100%;
    font-size: 12px; transition: background 0.2s;
  }}
  #node-info-panel .close-btn:hover {{ background: #e8e8e8; }}

  #help-text {{
    position: absolute; bottom: 14px; left: 50%; transform: translateX(-50%);
    background: rgba(255,255,255,0.92); padding: 7px 16px;
    border-radius: 20px; font-size: 11.5px; color: #666;
    box-shadow: 0 1px 4px rgba(0,0,0,0.1); z-index: 999;
    white-space: nowrap;
  }}
</style>

<div id="header">
  <h1>Sweet Recycler — BioProject × Ontology Network</h1>
  <div class="stats" id="stats">Loading…</div>
</div>

<div id="controls">
  <h3>Search Network</h3>
  <div class="search-wrapper">
    <input type="text" id="searchBox" placeholder="Search BioProject ID or term…">
    <div id="searchResults"></div>
  </div>

  <h3>Attribute Filter</h3>
  <select id="categorySelect" class="attribute-select" onchange="filterByAttribute(this.value)">
    <option value="ALL">Show All Categories</option>
    <option value="PO (anatomy)">PO (anatomy)</option>
    <option value="PO (stage)">PO (stage)</option>
    <option value="PECO">PECO (Condition)</option>
    <option value="TO">TO (Trait)</option>
    <option value="UO">UO (Unit)</option>
    <option value="BioProject">BioProject Nodes Only</option>
  </select>

  <h3>Categories</h3>
  <div id="legend"></div>

  <h3>View Controls</h3>
  <button class="btn" onclick="fitToView()">Fit to Screen</button>
  <button class="btn" onclick="resetView()">Reset Position</button>
  <button class="btn" onclick="togglePhysics()">Toggle Physics</button>
</div>

<div id="node-info-panel">
  <h3 id="ni-title">Node Info</h3>
  <div class="info-body" id="ni-body"></div>
  <button class="close-btn" onclick="closePanel()">Close</button>
</div>

<div id="help-text">
  Hover: highlight connections · Click node: inspect details · Attribute Filter: isolate categories · Scroll: zoom
</div>

<script>
  let physicsEnabled = true;
  // 100% Synced palette directly from Python's Okabe-Ito map
  const palette = {js_palette_json};

  function categoryColor(group) {{
    return palette[group] || "#999999";
  }}

  function buildLegend() {{
    const legend = document.getElementById("legend");
    legend.innerHTML = "";
    const cats = {{}};
    nodes.get().forEach(n => {{
      const g = n.group || "Other";
      if (!cats[g]) cats[g] = 0;
      cats[g]++;
    }});

    const ordered = ["BioProject"].concat(
      Object.keys(cats).filter(k => k !== "BioProject").sort()
    );
    ordered.forEach(name => {{
      if (!cats[name]) return;
      const item = document.createElement("div");
      item.className = "legend-item active";
      item.dataset.category = name;
      item.innerHTML =
        `<div class="legend-dot" style="background:${{categoryColor(name)}}"></div>` +
        `<span class="legend-label">${{name}}</span>` +
        `<span class="legend-count">${{cats[name]}}</span>`;
      item.onclick = () => toggleCategory(name, item);
      legend.appendChild(item);
    }});
  }}

  function toggleCategory(name, item) {{
    const isActive = item.classList.contains("active");
    const updates = nodes.get()
      .filter(n => n.group === name)
      .map(n => Object.assign({{}}, n, {{ hidden: isActive }}));
    nodes.update(updates);
    if (isActive) {{
      item.classList.remove("active");
      item.classList.add("inactive");
    }} else {{
      item.classList.add("active");
      item.classList.remove("inactive");
    }}
  }}

  function filterByAttribute(selectedGroup) {{
    if (selectedGroup === "ALL") {{
      const updates = nodes.get().map(n => Object.assign({{}}, n, {{ hidden: false }}));
      nodes.update(updates);
      document.querySelectorAll(".legend-item").forEach(el => {{
        el.classList.add("active");
        el.classList.remove("inactive");
      }});
      return;
    }}
    const updates = nodes.get().map(n => {{
      const matches = (n.group === selectedGroup);
      return Object.assign({{}}, n, {{ hidden: !matches }});
    }});
    nodes.update(updates);

    document.querySelectorAll(".legend-item").forEach(el => {{
      if (el.dataset.category === selectedGroup) {{
        el.classList.add("active");
        el.classList.remove("inactive");
      }} else {{
        el.classList.remove("active");
        el.classList.add("inactive");
      }}
    }});
  }}

  // ---- Interactive Dropdown Search -----------------------------------
  const searchBox = document.getElementById("searchBox");
  const searchResults = document.getElementById("searchResults");

  searchBox.addEventListener("input", function(e) {{
    const q = e.target.value.toLowerCase().trim();
    if (!q) {{
      searchResults.style.display = "none";
      return;
    }}
    const matches = nodes.get().filter(n =>
      (n.label || "").toLowerCase().includes(q) ||
      (n.id || "").toLowerCase().includes(q)
    ).slice(0, 10);

    if (matches.length > 0) {{
      searchResults.innerHTML = "";
      matches.forEach(m => {{
        const div = document.createElement("div");
        div.className = "search-item";
        const cat = m.group || "Other";
        div.innerHTML =
          `<span>${{m.label || m.id}}</span>` +
          `<span class="cat-tag" style="background:${{categoryColor(cat)}}">${{cat}}</span>`;
        div.onclick = () => selectAndFocusNode(m.id);
        searchResults.appendChild(div);
      }});
      searchResults.style.display = "block";
    }} else {{
      searchResults.style.display = "none";
    }}
  }});

  document.addEventListener("click", function(e) {{
    if (!e.target.closest(".search-wrapper")) {{
      searchResults.style.display = "none";
    }}
  }});

  function selectAndFocusNode(nodeId) {{
    searchResults.style.display = "none";
    network.focus(nodeId, {{
      scale: 0.9,
      animation: {{ duration: 600, easingFunction: "easeInOutQuad" }}
    }});
    network.selectNodes([nodeId]);
    const node = nodes.get(nodeId);
    document.getElementById("ni-title").innerText = node.label || nodeId;
    document.getElementById("ni-body").innerHTML = node.title || "No additional data.";
    document.getElementById("node-info-panel").style.display = "block";
  }}

  function fitToView() {{
    network.fit({{ animation: {{ duration: 600, easingFunction: "easeInOutQuad" }} }});
  }}
  function resetView() {{
    network.moveTo({{
      position: {{ x: 0, y: 0 }},
      scale: 1,
      animation: {{ duration: 600, easingFunction: "easeInOutQuad" }}
    }});
  }}
  function togglePhysics() {{
    physicsEnabled = !physicsEnabled;
    network.setOptions({{ physics: {{ enabled: physicsEnabled }} }});
  }}
  function closePanel() {{
    document.getElementById("node-info-panel").style.display = "none";
  }}

  network.on("click", function(params) {{
    const panel = document.getElementById("node-info-panel");
    if (params.nodes.length > 0) {{
      const nodeId = params.nodes[0];
      const node = nodes.get(nodeId);
      document.getElementById("ni-title").innerText =
        (node.label && node.label.trim() !== "") ? node.label : nodeId;
      document.getElementById("ni-body").innerHTML =
        node.title || "No additional data.";
      panel.style.display = "block";
    }} else {{
      panel.style.display = "none";
    }}
  }});

  function updateStats() {{
    const all = nodes.get();
    const bpCount = all.filter(n => n.group === "BioProject").length;
    const termCount = all.length - bpCount;
    document.getElementById("stats").textContent =
      `${{bpCount}} BioProjects · ${{termCount}} Terms · ${{edges.length}} Edges`;
  }}

  // Force immediate 0ms exact node color sync on initial page open
  nodes.update(nodes.get().map(n => ({{
    id: n.id,
    color: {{
      background: categoryColor(n.group),
      border: (n.group === "BioProject" ? "#888888" : "#1a1a1a"),
      highlight: {{ background: categoryColor(n.group), border: "#000000" }}
    }}
  }})));

  buildLegend();
  updateStats();
</script>
"""

html_content = html_content.replace("</body>", custom_ui + "\n</body>")
with open(html_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print("Saved:")
print(" - ../raw/figure2_ontology_terms.png / .pdf ")
print(" - ../raw/figure2_supplementary_interactive.html")

Saved:
 - ../raw/figure2_ontology_terms.png / .pdf 
 - ../raw/figure2_supplementary_interactive.html
